<a href="https://colab.research.google.com/github/Phionanamugga/Thesis/blob/feature1/pruning_energy_notebook_distilbertTF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Pruning Experiments with Energy Tracking (BERT / DistilBERT)
**What this notebook does:**

- Runs global magnitude pruning and head-level pruning experiments on BERT-style classifiers.

- Tracks **accuracy, latency, memory, and energy / CO₂** (via `codecarbon`).

- Repeats experiments multiple times and records mean ± std.

- Produces plots: *accuracy vs pruning amount*, *latency vs pruning amount*, *energy vs pruning amount*.

- Exports a pruned model to **ONNX** and runs **ONNX Runtime** (CPU) inference for more realistic edge CPU measurements.


> **Notes before running:**
- Best run in Google Colab or a machine with internet & GPU for speed. You may need to run the install cell.
- Energy estimates from `codecarbon` are approximate; dedicated power meters give more accurate readings.


Save location (after running): `/mnt/data/pruning_energy_results.csv` and `/mnt/data/pruning_energy_notebook.ipynb` (this file).


In [ ]:

# Install required packages (uncomment & run in Colab / local environment where pip is available)
!pip install -q transformers datasets sentencepiece codecarbon torch torchvision torchaudio psutil tqdm matplotlib pandas onnx onnxruntime

# If running in Colab, you may need to restart the runtime after torch install.
print('If running interactively, please install required packages first.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.7/277.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.34.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6

In [ ]:
# Imports and basic configuration
import os, time, copy, csv, math, random
import psutil
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
from evaluate import load as load_metric  # Updated for evaluate library
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch.nn.utils.prune as prune
from codecarbon import OfflineEmissionsTracker
import wandb
import onnx
import onnxruntime as ort

In [ ]:
# Explicitly set device to T4 GPU (Colab's default GPU)
DEVICE = torch.device("cuda:0")  # 'cuda:0' targets the first GPU (T4 in Colab)
print("Running on: T4 GPU")

Running on: T4 GPU


In [ ]:
# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
# General config - edit to your needs
MODEL_NAME = "distilbert-base-uncased"  # or 'distilbert-base-uncased'
TASK = "sst2"
BATCH_SIZE = 16
MAX_LENGTH = 128
PRUNING_LEVELS = [0.0, 0.2, 0.4, 0.6]
REPEATS = 3  # number of repeated runs per pruning level
WARMUP_ITERS = 20
RESULTS_CSV = "/content/pruning_energy_results.csv"  # Adjusted for Colab


In [ ]:
# Load SST-2 dataset and prepare dataloader
dataset = load_dataset("glue", "sst2")
metric = load_metric("accuracy")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def preprocess(batch):
    return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_dataset = dataset["train"].map(preprocess, batched=True)
eval_dataset = dataset["validation"].map(preprocess, batched=True)

# Keep only necessary columns
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in ["input_ids", "attention_mask", "label"]])
eval_dataset = eval_dataset.remove_columns([c for c in eval_dataset.column_names if c not in ["input_ids", "attention_mask", "label"]])

train_dataset.set_format(type="torch")
eval_dataset.set_format(type="torch")
eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=BATCH_SIZE)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(eval_dataset))


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Train samples: 67349
Validation samples: 872


In [ ]:

# Fine-tuning function with WandB monitoring
def fine_tune_model(model_name):
    print(f"Fine-tuning {model_name}...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"./{model_name}_fine_tuned",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        learning_rate=2e-5,
        seed=42,
        report_to="wandb",  # Log to WandB
        logging_steps=10
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset
    )

    # Start CodeCarbon tracking for fine-tuning
    tracker = OfflineEmissionsTracker(country_iso_code="DEU", save_to_file=False, log_level='info')
    tracker.start()

    trainer.train()

    emissions = tracker.stop()

    # Log fine-tuning emissions to WandB
    wandb.log({
        "fine_tuning_emissions_kg": emissions.emissions if hasattr(emissions, 'emissions') else emissions,
        "fine_tuning_total_energy_kwh": emissions.total_energy if hasattr(emissions, 'total_energy') else None
    })

    model.save_pretrained(f"./{model_name}_fine_tuned")
    return model

# Initialize WandB run and fine-tune the base model
wandb.init(project="fine_tuning_experiment", name="distilbert_sst2", config={"model_name": MODEL_NAME})
base_model = fine_tune_model(MODEL_NAME)
wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: phionanamugga23 (phionanamugga23-gisma-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[codecarbon INFO @ 19:56:52] offline tracker init
[codecarbon WARNING @ 19:56:52] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 19:56:52] [setup] RAM Tracking...
[codecarbon INFO @ 19:56:52] [setup] CPU Tracking...
[codecarbon WARNING @ 19:56:53] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 19:56:53] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 19:56:53] CPU Model on

Epoch,Training Loss,Validation Loss
1,0.116100,0.290868
2,0.130300,0.396091
3,0.077800,0.424056


[codecarbon INFO @ 19:57:08] Energy consumed for RAM : 0.000042 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 19:57:08] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 19:57:08] Energy consumed for All CPU : 0.000177 kWh
[codecarbon INFO @ 19:57:08] Energy consumed for all GPUs : 0.000267 kWh. Total GPU Power : 64.15229615331847 W
[codecarbon INFO @ 19:57:08] 0.000486 kWh of electricity used since the beginning.
[codecarbon INFO @ 19:57:23] Energy consumed for RAM : 0.000083 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 19:57:23] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 19:57:23] Energy consumed for All CPU : 0.000354 kWh
[codecarbon INFO @ 19:57:23] Energy consumed for all GPUs : 0.000554 kWh. Total GPU Power : 68.89381282942226 W
[codecarbon INFO @ 19:57:23] 0.000992 kWh of electricity used since the beginning.
[codecarbon INFO @ 19:57:38] Energy consumed for RAM : 0.000125 kWh. RAM Power : 1

eval/loss,▁▇█
eval/runtime,▅█▁
eval/samples_per_second,▄▁█
eval/steps_per_second,▄▁█
fine_tuning_emissions_kg,▁
train/epoch,▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇█
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train/grad_norm,▃▄▂▃▃▂▂▂▃▂█▁▄▄▆▁▄▁▂▁▆▁▁▁▆▁▂▃▁▁▁▁▇▁▁▃▁▂▅▁
train/learning_rate,███▇▇▇▇▇▇▇▆▅▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁
train/loss,█▇▅▅▇██▆▇▆▆▅▆▆▂▂▅▄▄▇▄▂▃▂▅▃▂▃▂▃▃▁▂▂▂▂▂▁▃▂
eval/loss,0.42406


In [ ]:
# Memory helper
process = psutil.Process(os.getpid())
def get_memory_info():
    rss_mb = process.memory_info().rss / (1024**2)  # MB
    gpu_mem_mb = None
    if torch.cuda.is_available():
        gpu_mem_mb = torch.cuda.memory_allocated() / (1024**2)
    return rss_mb, gpu_mem_mb


In [ ]:
# Fine-tune the base model
base_model = fine_tune_model(MODEL_NAME)

Fine-tuning distilbert-base-uncased...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[codecarbon INFO @ 20:42:45] offline tracker init
[codecarbon WARNING @ 20:42:45] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 20:42:45] [setup] RAM Tracking...
[codecarbon INFO @ 20:42:45] [setup] CPU Tracking...
[codecarbon WARNING @ 20:42:47] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 20:42:47] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 20:42:47] CPU Model on

Epoch,Training Loss,Validation Loss
1,0.103800,0.294603
2,0.135300,0.419740


[codecarbon INFO @ 20:43:02] Energy consumed for RAM : 0.000042 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 20:43:02] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 20:43:02] Energy consumed for All CPU : 0.000177 kWh
[codecarbon INFO @ 20:43:02] Energy consumed for all GPUs : 0.000252 kWh. Total GPU Power : 60.48926871071112 W
[codecarbon INFO @ 20:43:02] 0.000471 kWh of electricity used since the beginning.
[codecarbon INFO @ 20:43:17] Energy consumed for RAM : 0.000083 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 20:43:17] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 20:43:17] Energy consumed for All CPU : 0.000354 kWh
[codecarbon INFO @ 20:43:17] Energy consumed for all GPUs : 0.000539 kWh. Total GPU Power : 68.789272770285 W
[codecarbon INFO @ 20:43:17] 0.000976 kWh of electricity used since the beginning.
[codecarbon INFO @ 20:43:32] Energy consumed for RAM : 0.000125 kWh. RAM Power : 10.

Epoch,Training Loss,Validation Loss
1,0.103800,0.294603
2,0.135300,0.419740
3,0.079600,0.428450


[codecarbon INFO @ 21:25:16] Energy consumed for RAM : 0.007072 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 21:25:16] Delta energy consumed for CPU with constant : 0.000176 kWh, power : 42.5 W
[codecarbon INFO @ 21:25:16] Energy consumed for All CPU : 0.030067 kWh
[codecarbon INFO @ 21:25:16] Energy consumed for all GPUs : 0.042888 kWh. Total GPU Power : 50.485235018214006 W
[codecarbon INFO @ 21:25:16] 0.080026 kWh of electricity used since the beginning.
[codecarbon INFO @ 21:25:31] Energy consumed for RAM : 0.007113 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 21:25:31] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 21:25:31] Energy consumed for All CPU : 0.030244 kWh
[codecarbon INFO @ 21:25:31] Energy consumed for all GPUs : 0.043175 kWh. Total GPU Power : 68.88133667036045 W
[codecarbon INFO @ 21:25:31] 0.080532 kWh of electricity used since the beginning.
[codecarbon INFO @ 21:25:46] Energy consumed for RAM : 0.007155 kWh. RAM Power : 

In [ ]:
# Evaluation function (accuracy, latency, memory, energy) with WandB logging
def evaluate_model(model, dataloader, device, warmup_iters=WARMUP_ITERS, track_energy=True, pruning_type="none", prune_amount=0.0):
    model.eval()
    model.to(device)

    # Warm-up
    with torch.no_grad():
        it = iter(dataloader)
        for _ in range(min(warmup_iters, len(dataloader))):
            batch = next(it)
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            _ = model(input_ids=input_ids, attention_mask=attn)

    emissions = None
    if track_energy:
        tracker = OfflineEmissionsTracker(country_iso_code='DEU', save_to_file=False, log_level='info')
        tracker.start()

    total_correct = 0
    total_samples = 0
    total_time = 0.0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            t0 = time.perf_counter()
            outputs = model(input_ids=input_ids, attention_mask=attn)
            t1 = time.perf_counter()
            total_time += (t1 - t0)

            logits = outputs.logits
            preds = logits.argmax(dim=-1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    if track_energy:
        emissions = tracker.stop()

    avg_latency_ms = (total_time / total_samples) * 1000.0 if total_samples > 0 else None
    acc = total_correct / total_samples if total_samples > 0 else None
    cpu_rss_mb, gpu_mem_mb = get_memory_info()

    # Log evaluation metrics to WandB
    wandb.log({
        "eval_accuracy": acc,
        "eval_latency_ms": avg_latency_ms,
        "eval_cpu_memory_mb": cpu_rss_mb,
        "eval_gpu_memory_mb": gpu_mem_mb,
        "eval_emissions_kg": emissions.emissions if hasattr(emissions, 'emissions') else emissions,
        "eval_total_time_s": total_time,
        "pruning_type": pruning_type,
        "prune_amount": prune_amount
    })

    return {
        "accuracy": acc,
        "avg_latency_ms": avg_latency_ms,
        "cpu_rss_mb": cpu_rss_mb,
        "gpu_mem_mb": gpu_mem_mb,
        "emissions": emissions,
        "total_time_s": total_time,
        "total_samples": total_samples
    }

# Pruning utilities - global magnitude and head-zeroing
import copy as _copy

In [ ]:
def prune_model_global_magnitude(model, amount, prune_in_place=True):
    if not prune_in_place:
        model = _copy.deepcopy(model)

    to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            to_prune.append((module, 'weight'))

    prune.global_unstructured(parameters=to_prune, pruning_method=prune.L1Unstructured, amount=amount)

    # Make pruning permanent: remove reparam so zeros remain
    for module, _ in to_prune:
        try:
            prune.remove(module, 'weight')
        except Exception:
            pass
    return model

In [ ]:
def prune_attention_heads_by_zeroing(model, fraction_of_heads=0.2):
    model = _copy.deepcopy(model)
    config = model.config
    num_heads = getattr(config, 'num_attention_heads', None)
    hidden_size = getattr(config, 'hidden_size', None)
    if num_heads is None or hidden_size is None:
        print('Model does not expose num_attention_heads/hidden_size; skipping head pruning.')
        return model
    head_dim = hidden_size // num_heads
    heads_to_remove = int(math.ceil(num_heads * fraction_of_heads))
    for name, module in model.named_modules():
        if 'attention.self' in name and hasattr(module, 'out_features'):
            with torch.no_grad():
                weight = module.weight
                if weight.shape[0] == hidden_size:
                    start = (num_heads - heads_to_remove) * head_dim
                    end = num_heads * head_dim
                    weight[start:end, :] = 0.0
    return model

# Experiment loop: repeated runs per pruning level and aggregation of mean/std
from statistics import mean, stdev

In [ ]:
def run_pruning_experiments(model_name=MODEL_NAME, pruning_levels=PRUNING_LEVELS, repeats=REPEATS, results_csv=RESULTS_CSV):
    # Initialize WandB for the experiment
    wandb.init(project="pruning_experiments", name=f"{model_name}_pruning", config={
        "model_name": model_name,
        "pruning_levels": pruning_levels,
        "repeats": repeats,
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH
    })

    base_model = AutoModelForSequenceClassification.from_pretrained(f"./{model_name}_fine_tuned", num_labels=2)
    base_model.to(DEVICE)
    base_model.eval()

    # CSV header
    header = ['model','pruning_type','prune_amount','run_idx','accuracy','avg_latency_ms','cpu_rss_mb','gpu_mem_mb','emissions','total_time_s','total_samples']
    if not os.path.exists(results_csv):
        with open(results_csv, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(header)

    results = []

    # Global magnitude pruning
    for amount in pruning_levels:
        for run_idx in range(repeats):
            print(f'Pruning global magnitude amount={amount}, run {run_idx+1}/{repeats}')
            model_pruned = prune_model_global_magnitude(_copy.deepcopy(base_model), amount, prune_in_place=True)
            res = evaluate_model(model_pruned, eval_loader, DEVICE, track_energy=True, pruning_type='global_magnitude', prune_amount=amount)

            emissions_str = str(res['emissions']) if res['emissions'] is not None else ''
            row = [model_name, 'global_magnitude', amount, run_idx, res['accuracy'], res['avg_latency_ms'], res['cpu_rss_mb'], res['gpu_mem_mb'], emissions_str, res['total_time_s'], res['total_samples']]
            with open(results_csv, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row)

            results.append(row)

            # Log to WandB
            wandb.log({
                "global_magnitude_accuracy": res['accuracy'],
                "global_magnitude_latency_ms": res['avg_latency_ms'],
                "global_magnitude_emissions_kg": res['emissions'].emissions if hasattr(res['emissions'], 'emissions') else res['emissions'],
                "prune_amount": amount,
                "run_idx": run_idx
            })

    # Head-pruning repeats
    head_fractions = [0.1, 0.2, 0.4]
    for fraction in head_fractions:
        for run_idx in range(repeats):
            print(f'Head-pruning fraction={fraction}, run {run_idx+1}/{repeats}')
            model_hp = prune_attention_heads_by_zeroing(_copy.deepcopy(base_model), fraction_of_heads=fraction)
            res = evaluate_model(model_hp, eval_loader, DEVICE, track_energy=True, pruning_type='head_zeroing', prune_amount=fraction)

            emissions_str = str(res['emissions']) if res['emissions'] is not None else ''
            row = [model_name, 'head_zeroing', fraction, run_idx, res['accuracy'], res['avg_latency_ms'], res['cpu_rss_mb'], res['gpu_mem_mb'], emissions_str, res['total_time_s'], res['total_samples']]
            with open(results_csv, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row)

            results.append(row)

            # Log to WandB
            wandb.log({
                "head_zeroing_accuracy": res['accuracy'],
                "head_zeroing_latency_ms": res['avg_latency_ms'],
                "head_zeroing_emissions_kg": res['emissions'].emissions if hasattr(res['emissions'], 'emissions') else res['emissions'],
                "prune_fraction": fraction,
                "run_idx": run_idx
            })

    wandb.finish()
    print('All experiments recorded to', results_csv)
    return results_csv

In [ ]:
import pandas as pd
import os
print(os.path.exists(results_csv))  # Should print True if the file exists
df = pd.read_csv(results_csv)
print(df.head())  # Inspect the first few rows of the DataFrame
print(df.shape)   # Check the number of rows and columns

NameError: name 'results_csv' is not defined

In [ ]:
print(os.path.exists(results_csv))

NameError: name 'results_csv' is not defined

In [ ]:
# Aggregation and plotting of results (after experiments complete)
def aggregate_and_plot(results_csv=RESULTS_CSV, output_prefix='/content/pruning_plots.csv'):
    df = pd.read_csv(results_csv)

    # Convert emissions field to numeric where possible (attempt to parse 'emissions_kg' or 'emissions' keys)
    def extract_emissions(s):
        if pd.isna(s) or s == '':
            return np.nan
        try:
            d = eval(s)
            if isinstance(d, dict):
                for k in ['emissions_kg','emissions','energy_consumed']:
                    if k in d:
                        return float(d[k])
            return np.nan
        except Exception:
            return np.nan
    df['emissions_val'] = df['emissions'].apply(extract_emissions)

    summary = df.groupby(['pruning_type','prune_amount']).agg({
        'accuracy': ['mean','std'],
        'avg_latency_ms': ['mean','std'],
        'cpu_rss_mb': ['mean','std'],
        'gpu_mem_mb': ['mean','std'],
        'emissions_val': ['mean','std']
    }).reset_index()

    # Flatten columns
    summary.columns = ['_'.join([str(c) for c in col]).strip('_') for col in summary.columns.values]

    print("Summary Statistics:")
    print(summary.head(20))

    # Plot: accuracy vs prune amount
    for metric_name, col_mean, col_std in [
        ('Accuracy', 'accuracy_mean', 'accuracy_std'),
        ('Latency (ms)', 'avg_latency_ms_mean', 'avg_latency_ms_std'),
        ('Emissions (kg CO2 eq)', 'emissions_val_mean', 'emissions_val_std')
    ]:
        plt.figure(figsize=(10, 6))
        for ptype in summary['pruning_type_'].unique():
            sub = summary[summary['pruning_type_']==ptype]
            x = sub['prune_amount_']
            y = sub[col_mean]
            yerr = sub[col_std].fillna(0)
            plt.errorbar(x, y, yerr=yerr, label=ptype, marker='o', capsize=4, linewidth=2)
        plt.xlabel('Prune amount / fraction')
        plt.ylabel(metric_name)
        plt.title(f'{metric_name} vs Prune amount (with error bars)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        out_file = f"{output_prefix}_{metric_name.replace(' ','_').replace(' (kg CO2 eq)', '')}.png"
        plt.savefig(out_file, dpi=300, bbox_inches='tight')
        print(f'Saved plot to {out_file}')
        plt.show()

    # Radar chart for multi-metric comparison across pruning types
    def radar_chart_summary(summary):
        # Select representative pruning levels for radar chart
        metrics = ['accuracy_mean', 'avg_latency_ms_mean', 'emissions_val_mean']
        metric_labels = ['Accuracy', 'Latency (ms)', 'Emissions (kg)']

        # Normalize metrics (higher is better for accuracy, lower for others)
        summary_norm = summary.copy()
        for m in metrics:
            if m == 'accuracy_mean':
                summary_norm[m] = summary_norm[m] / summary_norm[m].max()
            else:
                summary_norm[m] = 1 / (summary_norm[m] / summary_norm[m].min()) if summary_norm[m].min() > 0 else 0

        N = len(metrics)
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
        angles += angles[:1]

        plt.figure(figsize=(10, 10))
        ax = plt.subplot(111, polar=True)

        # Plot for each pruning type at a fixed prune amount (e.g., 0.4)
        for ptype in summary_norm['pruning_type_'].unique():
            sub = summary_norm[summary_norm['pruning_type_'] == ptype]
            if len(sub) > 0:
                row = sub.iloc[0]  # Take first row for representative
                values = [row[m] for m in metrics]
                values += values[:1]
                ax.plot(angles, values, linewidth=3, label=ptype)
                ax.fill(angles, values, alpha=0.25)

        plt.xticks(angles[:-1], metric_labels, size=12)
        plt.title("Pruning Method Comparison (Radar Chart) - Energy Focus", size=16, pad=20)
        ax.set_rlabel_position(0)
        plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"], size=10)
        plt.legend(loc="upper right", bbox_to_anchor=(0.1, 0.1))
        plt.grid(True)

        # Highlight energy usage
        ax.text(angles[2] * 0.9, 1.1, "Emissions", ha="center", va="center", color="red", fontsize=14)

        plt.tight_layout()
        radar_file = f"{output_prefix}_radar.png"
        plt.savefig(radar_file, dpi=300, bbox_inches='tight')
        print(f'Radar chart saved to {radar_file}')
        plt.show()

    # Generate radar chart
    radar_chart_summary(summary)

    return summary

In [ ]:
# Run experiments
results_csv = run_pruning_experiments()

# Aggregate and plot results
summary = aggregate_and_plot(results_csv)

# ONNX export + ONNX Runtime (CPU) inference for pruned model
def export_to_onnx(model, tokenizer, export_path='/content/pruned_model.onnx', sample_text='This is a test sentence.', device=DEVICE):
    model.to('cpu')  # export on CPU to simplify
    model.eval()
    tokens = tokenizer(sample_text, return_tensors='pt', truncation=True, padding='max_length', max_length=MAX_LENGTH)
    input_names = ['input_ids', 'attention_mask']
    output_names = ['logits']
    torch.onnx.export(model, (tokens['input_ids'], tokens['attention_mask']), export_path,
                      input_names=input_names, output_names=output_names,
                      opset_version=12, dynamic_axes={'input_ids': {0: 'batch_size'}, 'attention_mask': {0:'batch_size'}})
    print('ONNX exported to', export_path)

    # Log ONNX export to WandB
    wandb.log({"onnx_exported": wandb.Info({"path": export_path})})

    return export_path

def run_onnx_inference(onnx_path, tokenizer, texts, batch_size=8):
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    input_name_ids = sess.get_inputs()[0].name
    input_name_mask = sess.get_inputs()[1].name
    results = []
    tracker = OfflineEmissionsTracker(country_iso_code="DEU", save_to_file=False, log_level='info')
    tracker.start()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        toks = tokenizer(batch_texts, truncation=True, padding='max_length', max_length=MAX_LENGTH, return_tensors='np')
        ort_inputs = {input_name_ids: toks['input_ids'], input_name_mask: toks['attention_mask']}
        t0 = time.perf_counter()
        out = sess.run(None, ort_inputs)
        t1 = time.perf_counter()
        logits = out[0]
        preds = np.argmax(logits, axis=-1)
        results.extend(preds.tolist())

    emissions = tracker.stop()

    # Log ONNX inference to WandB
    wandb.log({
        "onnx_inference_time_s": time.perf_counter() - t0,
        "onnx_emissions_kg": emissions.emissions if hasattr(emissions, 'emissions') else emissions
    })

    return results

# Example usage after running experiments:
# Load a pruned model for ONNX export (example with global magnitude pruning 0.4)
pruned_model = prune_model_global_magnitude(_copy.deepcopy(base_model), amount=0.4, prune_in_place=True)
export_path = export_to_onnx(pruned_model, tokenizer, export_path='/content/pruned_model.onnx')
preds = run_onnx_inference(export_path, tokenizer, ['I love this movie', 'I hate this movie'])
print("ONNX predictions:", preds)


## How to run
1. (Optional) Install required packages in a notebook cell (see install cell). If running in Colab, run the install cell and then **restart runtime** if necessary.

2. Run cells in order.

3. To execute experiments, run:

```python
results_file = run_pruning_experiments()
```

4. After experiments finish, aggregate and plot:

```python
summary = aggregate_and_plot(results_file)
```

5. To export to ONNX and run ONNX Runtime CPU inference follow the ONNX cell examples.


**Reproducibility tips:** record environment info (torch, cuda, CPU/GPU models) and run each experiment multiple times.


In [1]:
!pip install --upgrade nbformat nbconvert ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.0 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [3]:
!pip install jupyter nbconvert

  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 6.4 MB/s eta 0:00:00
